# WACTRA: Wavelet-Adversarial Cross-domain Temporal Risk Attribution
**AI-Based Detection of Hardware Side-Channel Leakage and Cryptographic Manipulation in IoT Devices**

| Stage | Method | Contribution |
|---|---|---|
| 1 | Data collection: Chameleon + UCI hardware-Trojan power/EM = **IoTCSHTD** | |
| 2 | Robust scaling, label encoding, annotation-based segmentation, acquisition-level splits | |
| 3 | Wavelet Packet Decomposition | **C1 OAT-SLD** |
| 4 | Temporal Convolutional Network (side-channel leakage detection) | |
| 5 | Domain-adversarial learning (hardware-Trojan detection) | **C2 CDL-HMD** |
| 6 | Multi-task learning + calibrated joint decision | |
| 7 | SHAP attribution + threat-specific risk scores | **C3 XH-TRA** |

**Data note.** Put the real datasets in `data/chameleon` and `data/uci_ht` (see Stage 1 for the expected keys; adjust `CFG` to match your files).
If those folders are absent, the notebook generates a *synthetic stand-in* (AES S-box Hamming-weight leakage with BASE/DFS/RD/MRP/CHF-style countermeasures, and a Trojan power/EM dataset), so the whole pipeline runs end to end. **Numbers from synthetic data only validate the code, not the research claims.**

In [ ]:
!pip install -q pywavelets shap h5py scikit-learn pandas matplotlib
# torch is preinstalled on Colab / Kaggle. Locally: !pip install -q torch

In [ ]:
import os, math, copy, json, time, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import pywt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, precision_score,
                             recall_score, confusion_matrix, classification_report)
from scipy.stats import spearmanr
try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False
warnings.filterwarnings("ignore")

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

CFG = dict(
    QUICK=False,                 # True = tiny/fast smoke test
    WINDOW=1024, WAVELET="db4", WPD_LEVEL=3,
    # ---- real data locations / key names (edit to match your files) ----
    CHAM_DIR="data/chameleon", UCI_DIR="data/uci_ht",
    CHAM_KEYS=dict(traces="traces", condition="condition", start="start", end="end", group="acquisition"),
    UCI_KEYS=dict(power="power", em="em", label="label", group="chip"),
    LEAKY_CONDITIONS=("BASE",),  # conditions treated as 'exploitable leakage present'
    TROJAN_STRINGS=("trojan", "ht", "infected", "1", "true"),
    # ---- synthetic stand-in sizes ----
    N_CHAM_PER_COND=500, N_CHIPS=48, N_PER_CHIP=30,
    # ---- training ----
    BATCH=64, LR=2e-3, WD=1e-4, EPOCHS_S4=30, EPOCHS_S5=40, PATIENCE=8,
    DANN_LAMBDA=1.0,
    # ---- risk ----
    SEVERITY=dict(leakage=0.7, hardware_modification=1.0),
)
if CFG["QUICK"]:
    CFG.update(N_CHAM_PER_COND=100, N_CHIPS=24, N_PER_CHIP=20, EPOCHS_S4=4, EPOCHS_S5=5)
print("device:", DEVICE, "| shap:", HAS_SHAP)

## Stage 1: Data collection (IoTCSHTD)
Loader accepts `.npz`, `.h5/.hdf5`, `.csv`, `.npy` files inside each folder (every array is registered by its key / file stem).
Expected arrays:
* **Chameleon**: `traces` (N, T_raw) AES power traces, `condition` (BASE/DFS/RD/MRP/CHF), optional `start`/`end` (cryptographic-operation annotations), optional `acquisition` id.
* **UCI hardware-Trojan**: `power` (M, T), `em` (M, T), `label` (normal/Trojan), optional `chip` id.

Dataset identity is kept separate throughout (two dicts, never merged into one label space).

In [ ]:
def read_container(path):
    path = Path(path); s = path.suffix.lower()
    if s == ".npz":
        z = np.load(path, allow_pickle=True); return {k: z[k] for k in z.files}
    if s in (".h5", ".hdf5"):
        import h5py
        out = {}
        with h5py.File(path, "r") as f:
            def visit(name, obj):
                if isinstance(obj, h5py.Dataset): out[name.split("/")[-1]] = obj[()]
            f.visititems(visit)
        return out
    if s == ".csv":
        df = pd.read_csv(path); return {c: df[c].values for c in df.columns}
    if s == ".npy":
        return {path.stem: np.load(path, allow_pickle=True)}
    return {}

def collect_dir(d):
    out = {}
    for p in sorted(Path(d).rglob("*")):
        if p.is_file(): out.update(read_container(p))
    return out

def load_real_chameleon():
    d = collect_dir(CFG["CHAM_DIR"]); K = CFG["CHAM_KEYS"]
    for need in ("traces", "condition"):
        if K[need] not in d: raise KeyError(f"Chameleon key '{K[need]}' not found; available: {list(d)[:20]}")
    return dict(traces=np.asarray(d[K["traces"]], dtype=np.float64), cond=np.asarray(d[K["condition"]]),
                start=d.get(K["start"]), end=d.get(K["end"]), group=d.get(K["group"]))

def load_real_uci():
    d = collect_dir(CFG["UCI_DIR"]); K = CFG["UCI_KEYS"]
    for need in ("power", "em", "label"):
        if K[need] not in d: raise KeyError(f"UCI key '{K[need]}' not found; available: {list(d)[:20]}")
    return dict(power=np.asarray(d[K["power"]], dtype=np.float64), em=np.asarray(d[K["em"]], dtype=np.float64),
                label=np.asarray(d[K["label"]]), group=d.get(K["group"]))

# ---------------- synthetic stand-ins (only used when real data is absent) ----------------
def make_sbox():
    sbox = [0] * 256; p = q = 1
    rotl = lambda x, s: ((x << s) | (x >> (8 - s))) & 0xFF
    while True:
        p = (p ^ ((p << 1) & 0xFF) ^ (0x1B if p & 0x80 else 0)) & 0xFF
        q ^= (q << 1) & 0xFF; q ^= (q << 2) & 0xFF; q ^= (q << 4) & 0xFF
        if q & 0x80: q ^= 0x09
        sbox[p] = (q ^ rotl(q, 1) ^ rotl(q, 2) ^ rotl(q, 3) ^ rotl(q, 4) ^ 0x63) & 0xFF
        if p == 1: break
    sbox[0] = 0x63
    return np.array(sbox, dtype=np.int64)
SBOX = make_sbox(); assert SBOX[1] == 0x7C and SBOX[0x53] == 0xED
POPCNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.float64)
CONDS = ["BASE", "DFS", "RD", "MRP", "CHF"]

def _burst(Le, centers, amps):
    t = np.arange(Le, dtype=np.float64)[None, :]; c = centers[:, None]
    pulse = np.exp(-((t - c) / 6.0) ** 2 / 2) - 0.4 * np.exp(-((t - c - 12) / 10.0) ** 2 / 2)
    env = 0.5 * np.sin(np.pi * np.arange(Le) / Le)
    return env + (amps[:, None] * pulse).sum(0)

def synth_chameleon(n_per_cond, key, rng, per_acq=25, Lraw=1800, Le0=960):
    traces, conds, starts, ends, groups = [], [], [], [], []; acq = 0
    for cond in CONDS:
        for a in range(max(1, n_per_cond // per_acq)):
            gain, off, sig = rng.uniform(0.8, 1.2), rng.normal(0, 0.1), rng.uniform(0.25, 0.4)
            for _ in range(per_acq):
                pt = rng.integers(0, 256, 16); v = SBOX[pt ^ key]
                centers = (np.arange(16) + 0.5) * Le0 / 16.0
                if cond == "MRP": v = v ^ rng.integers(0, 256, 16)       # randomised masking decorrelates HW
                amps = POPCNT[v] * 0.25
                if cond == "RD": centers = centers + np.cumsum(rng.integers(0, 36, 16) * (rng.random(16) < 0.6))
                Le = int(centers[-1] + 60)
                b = _burst(Le, centers, amps)
                if cond == "DFS":
                    s = rng.uniform(0.75, 1.3); n2 = int(Le * s)
                    b = np.interp(np.linspace(0, Le - 1, n2), np.arange(Le), b) * s ** -0.5; Le = n2
                if cond == "CHF":
                    A, f, ph = rng.uniform(10, 40), rng.uniform(2, 6), rng.uniform(0, 6.28)
                    tw = np.clip(np.arange(Le) + A * np.sin(2 * np.pi * f * np.arange(Le) / Le + ph), 0, Le - 1)
                    b = np.interp(tw, np.arange(Le), b) + 0.15 * np.sin(2 * np.pi * 0.11 * np.arange(Le) + ph)
                if cond == "MRP": b = b + rng.normal(0, 0.2, Le)
                s0 = int(rng.integers(60, 200)); x = rng.normal(off, sig, Lraw) * 0.6
                x[s0:s0 + Le] += gain * b; x += rng.normal(0, sig, Lraw)
                traces.append(x); conds.append(cond); starts.append(s0); ends.append(s0 + Le); groups.append(acq)
            acq += 1
    return dict(traces=np.array(traces), cond=np.array(conds), start=np.array(starts),
                end=np.array(ends), group=np.array(groups))

def synth_uci(n_chips, n_per_chip, rng, L=1024):
    P, E, y, g = [], [], [], []; t = np.arange(L)
    for c in range(n_chips):
        troj = int(c % 2); gp, ge = rng.uniform(0.8, 1.25), rng.uniform(0.7, 1.4)
        f1, f2 = 0.02 + rng.normal(0, 0.002), 0.07 + rng.normal(0, 0.003)
        for _ in range(n_per_chip):
            ph = rng.uniform(0, 6.28, 3)
            base = np.sin(2*np.pi*f1*t + ph[0]) + 0.5*np.sin(2*np.pi*f2*t + ph[1]) + 0.3*np.sin(2*np.pi*0.15*t + ph[2])
            p = gp * base + rng.normal(0, 0.3, L); e = ge * np.gradient(base) * 8 + rng.normal(0, 0.4, L)
            if troj:
                pos = int(rng.integers(100, L - 250))
                p[pos:pos+150] += 0.35 * np.sin(2*np.pi*0.35*t[:150] + rng.uniform(0, 6.28))
                e += 0.18 * np.sin(2*np.pi*0.21*t + rng.uniform(0, 6.28))
            P.append(p); E.append(e); y.append(troj); g.append(c)
    return dict(power=np.array(P), em=np.array(E), label=np.array(y), group=np.array(g))

def get_datasets():
    rng = np.random.default_rng(SEED); key = rng.integers(0, 256, 16)
    ok_c = Path(CFG["CHAM_DIR"]).exists() and any(Path(CFG["CHAM_DIR"]).iterdir())
    ok_u = Path(CFG["UCI_DIR"]).exists() and any(Path(CFG["UCI_DIR"]).iterdir())
    cham = load_real_chameleon() if ok_c else synth_chameleon(CFG["N_CHAM_PER_COND"], key, rng)
    uci = load_real_uci() if ok_u else synth_uci(CFG["N_CHIPS"], CFG["N_PER_CHIP"], rng)
    cham["source"] = "real" if ok_c else "SYNTHETIC"; uci["source"] = "real" if ok_u else "SYNTHETIC"
    return cham, uci

raw_cham, raw_uci = get_datasets()
print("Chameleon:", raw_cham["source"], raw_cham["traces"].shape, "| conditions:", pd.Series(raw_cham["cond"]).value_counts().to_dict())
print("UCI-HT   :", raw_uci["source"], raw_uci["power"].shape, "| labels:", pd.Series(raw_uci["label"]).value_counts().to_dict())

## Stage 2: Preprocessing and label preparation
Robust scaling (median / IQR from **training** data only, per channel), label encoding, removal of incomplete records,
annotation-based extraction of the cryptographic segment, separate labels per dataset, acquisition/chip-grouped splits.

In [ ]:
def crop_resample(x, s, e, L):
    seg = x[int(s):int(e)] if (s is not None and e is not None) else x
    if len(seg) < 8: return None
    return np.interp(np.linspace(0, len(seg) - 1, L), np.arange(len(seg)), seg)

def preprocess_chameleon(d, L):
    n = len(d["traces"]); keep, X = [], []
    for i in range(n):
        x = d["traces"][i]
        if not np.all(np.isfinite(x)): continue
        s = d["start"][i] if d["start"] is not None else None
        e = d["end"][i] if d["end"] is not None else None
        if s is not None and e is not None and not (np.isfinite(s) and np.isfinite(e) and e > s): continue
        w = crop_resample(x, s, e, L)
        if w is None: continue
        keep.append(i); X.append(w)
    keep = np.array(keep); X = np.array(X, dtype=np.float32)[:, None, :]
    cond = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in d["cond"][keep]])
    le = LabelEncoder().fit(cond); yc = le.transform(cond)
    yl = np.isin(cond, CFG["LEAKY_CONDITIONS"]).astype(np.int64)
    grp = d["group"][keep] if d["group"] is not None else np.arange(len(keep)) // 25
    return X, yc, yl, np.asarray(grp), le

def preprocess_uci(d, L):
    ok = np.isfinite(d["power"]).all(1) & np.isfinite(d["em"]).all(1)
    def rs(A): return np.array([np.interp(np.linspace(0, A.shape[1]-1, L), np.arange(A.shape[1]), a) for a in A[ok]], dtype=np.float32)
    X = np.stack([rs(d["power"]), rs(d["em"])], 1)
    lab = np.asarray(d["label"])[ok]
    if lab.dtype.kind in "OUS":
        lab = np.array([(v.decode() if isinstance(v, bytes) else str(v)).lower() in CFG["TROJAN_STRINGS"] for v in lab]).astype(np.int64)
    else:
        lab = (lab != 0).astype(np.int64)
    grp = d["group"][ok] if d["group"] is not None else np.arange(ok.sum()) // 25
    return X, lab, np.asarray(grp)

def group_split3(y, groups, seed=SEED, fr=(0.70, 0.15, 0.15)):
    # acquisition-level grouping; retry until every split contains all classes
    classes = np.unique(y)
    for k in range(200):
        g1 = GroupShuffleSplit(1, test_size=fr[2] + fr[1], random_state=seed + k)
        tr, rest = next(g1.split(np.zeros(len(y)), y, groups))
        g2 = GroupShuffleSplit(1, test_size=fr[2] / (fr[1] + fr[2]), random_state=seed + k)
        va_i, te_i = next(g2.split(np.zeros(len(rest)), y[rest], groups[rest]))
        va, te = rest[va_i], rest[te_i]
        if all(len(np.unique(y[s])) == len(classes) for s in (tr, va, te)):
            assert not (set(groups[tr]) & set(groups[te])) and not (set(groups[tr]) & set(groups[va]))
            return tr, va, te
    raise RuntimeError("could not build splits containing all classes; check group ids")

class RobustScaler1D:
    # per-channel median / IQR fitted on training data only
    def fit(self, X):
        C = X.shape[1]; self.med = np.zeros((1, C, 1), np.float32); self.iqr = np.ones((1, C, 1), np.float32)
        for c in range(C):
            v = X[:, c].ravel(); v = v[np.random.default_rng(0).integers(0, len(v), min(len(v), 500000))]
            q1, m, q3 = np.percentile(v, [25, 50, 75]); self.med[0, c, 0] = m; self.iqr[0, c, 0] = max(q3 - q1, 1e-8)
        return self
    def transform(self, X): return np.clip((X - self.med) / self.iqr, -20, 20).astype(np.float32)

L = CFG["WINDOW"]
Xc_raw, yc, yl, gc, le_cond = preprocess_chameleon(raw_cham, L)
Xu_raw, yu, gu = preprocess_uci(raw_uci, L)
print("Chameleon classes:", dict(zip(le_cond.classes_, np.bincount(yc))), "| leaky positives:", int(yl.sum()), "/", len(yl))
print("UCI Trojan share :", round(float(yu.mean()), 3), "| samples:", len(yu))

tr_c, va_c, te_c = group_split3(yc, gc)
tr_u, va_u, te_u = group_split3(yu, gu)
sc_c = RobustScaler1D().fit(Xc_raw[tr_c]); sc_u = RobustScaler1D().fit(Xu_raw[tr_u])
Xc = sc_c.transform(Xc_raw); Xu = sc_u.transform(Xu_raw)
print("Chameleon split:", len(tr_c), len(va_c), len(te_c), "| UCI split:", len(tr_u), len(va_u), len(te_u))

## Stage 3 (Contribution 1, OAT-SLD): Wavelet Packet Decomposition
Each cryptographic window is decomposed into 2^level frequency-ordered subbands (Daubechies-4, periodization mode so every subband has length T/2^level).
The stacked subband sequences form the unified temporal representation fed to the TCN. For UCI, power and EM are decomposed separately (8 + 8 channels).

In [ ]:
def wpd_freq(x, wavelet, level, rev=False):
    # recursive packet tree returning subbands in true frequency order
    if level == 0: return [x]
    a, d = pywt.dwt(x, wavelet, mode="periodization", axis=-1)
    la = wpd_freq(a, wavelet, level - 1, rev)
    ld = wpd_freq(d, wavelet, level - 1, not rev)
    return ld + la if rev else la + ld

def wpd_tensor(X, wavelet=None, level=None):
    wavelet = wavelet or CFG["WAVELET"]; level = level or CFG["WPD_LEVEL"]
    N, C, T = X.shape; chans = []
    for c in range(C):
        chans += wpd_freq(X[:, c].astype(np.float64), wavelet, level)
    return np.stack(chans, 1).astype(np.float32)            # (N, C*2^level, T/2^level)

NB = 2 ** CFG["WPD_LEVEL"]
Fc = wpd_tensor(Xc); Fu = wpd_tensor(Xu)
print("WPD tensors -> Chameleon:", Fc.shape, "| UCI:", Fu.shape)

def band_label(b): return f"{b/(2*NB):.3f}-{(b+1)/(2*NB):.3f}"
BAND_NAMES = [f"B{b} ({band_label(b)} cyc/smp)" for b in range(NB)]

# --- compare execution conditions in the wavelet-packet domain ---
logE = np.log(np.mean(Fc[tr_c] ** 2, axis=2) + 1e-8)               # (N, 8)
cond_names = list(le_cond.classes_)
M = np.stack([logE[yc[tr_c] == k].mean(0) for k in range(len(cond_names))])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
im = ax[0].imshow(M, aspect="auto", cmap="viridis"); ax[0].set_yticks(range(len(cond_names))); ax[0].set_yticklabels(cond_names)
ax[0].set_xticks(range(NB)); ax[0].set_xlabel("WPD subband (low -> high frequency)"); ax[0].set_title("Mean log-energy per condition"); plt.colorbar(im, ax=ax[0])
Mz = (M - M.mean(0, keepdims=True)) / (M.std(0, keepdims=True) + 1e-8)
ax[1].imshow(Mz, aspect="auto", cmap="coolwarm"); ax[1].set_yticks(range(len(cond_names))); ax[1].set_yticklabels(cond_names)
ax[1].set_xticks(range(NB)); ax[1].set_title("Per-band z-score across conditions (obfuscation signature)")
plt.tight_layout(); plt.savefig(OUT / "c1_wpd_condition_signatures.png", dpi=130); plt.show()

# --- sanity check: WPD band ordering on a known tone ---
tone = np.sin(2 * np.pi * 0.35 * np.arange(1024))[None, :]
en = [np.mean(b ** 2) for b in wpd_freq(tone, CFG["WAVELET"], CFG["WPD_LEVEL"])]
print("0.35 cyc/sample tone peaks in subband", int(np.argmax(en)), "(expected 5)")

def subband_stats(Fx):
    return np.concatenate([np.log(np.mean(Fx ** 2, 2) + 1e-8), np.mean(np.abs(Fx), 2), np.max(np.abs(Fx), 2)], 1)
Sc, Su = subband_stats(Fc), subband_stats(Fu)

## Models
Shared TCN encoder with per-dataset 1x1 input adapters (the two datasets have different channel layouts) and three separate heads
(execution condition, side-channel leakage, hardware Trojan) plus a domain discriminator behind a gradient-reversal layer.

In [ ]:
class Block(nn.Module):
    def __init__(s, ci, co, k, d, p):
        super().__init__(); s.pad = (k - 1) * d
        s.c1 = nn.Conv1d(ci, co, k, dilation=d); s.n1 = nn.GroupNorm(4, co)
        s.c2 = nn.Conv1d(co, co, k, dilation=d); s.n2 = nn.GroupNorm(4, co)
        s.drop = nn.Dropout(p); s.res = nn.Conv1d(ci, co, 1) if ci != co else nn.Identity()
    def forward(s, x):
        h = s.drop(F.relu(s.n1(s.c1(F.pad(x, (s.pad, 0))))))
        h = s.drop(F.relu(s.n2(s.c2(F.pad(h, (s.pad, 0))))))
        return F.relu(h + s.res(x))

class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam): ctx.lam = lam; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.lam * g, None

class WACTRA(nn.Module):
    def __init__(s, ch_c, ch_u, n_cond, hid=64, p=0.2, chans=(48, 64, 64, 64)):
        super().__init__()
        s.adapt = nn.ModuleDict(dict(c=nn.Conv1d(ch_c, 32, 1), u=nn.Conv1d(ch_u, 32, 1)))
        blocks, ci = [], 32
        for i, co in enumerate(chans): blocks.append(Block(ci, co, 3, 2 ** i, p)); ci = co
        s.tcn = nn.Sequential(*blocks)
        s.proj = nn.Sequential(nn.Linear(2 * ci, hid), nn.ReLU(), nn.Dropout(p))
        s.head_cond = nn.Linear(hid, n_cond); s.head_leak = nn.Linear(hid, 1); s.head_troj = nn.Linear(hid, 1)
        s.disc = nn.Sequential(nn.Linear(hid, 64), nn.ReLU(), nn.Linear(64, 1))
        s.log_var = nn.Parameter(torch.zeros(3))               # uncertainty weights for multi-task loss
    def embed(s, x, domain):
        h = s.tcn(s.adapt[domain](x))
        return s.proj(torch.cat([h.mean(-1), h.amax(-1)], 1))

CH_C, CH_U, N_COND = Fc.shape[1], Fu.shape[1], len(le_cond.classes_)
def new_model(): return WACTRA(CH_C, CH_U, N_COND).to(DEVICE)
print("parameters:", sum(p.numel() for p in new_model().parameters()))

@torch.no_grad()
def predict(model, X, domain, bs=256):
    model.eval(); o = dict(cond=[], leak=[], troj=[], emb=[])
    for i in range(0, len(X), bs):
        z = model.embed(torch.as_tensor(X[i:i + bs]).to(DEVICE), domain)
        o["cond"].append(model.head_cond(z).cpu()); o["leak"].append(model.head_leak(z).squeeze(1).cpu())
        o["troj"].append(model.head_troj(z).squeeze(1).cpu()); o["emb"].append(z.cpu())
    return {k: torch.cat(v).numpy() for k, v in o.items()}

sigmoid = lambda a: 1 / (1 + np.exp(-a))
def bin_metrics(y, p, thr=0.5):
    pr = (p >= thr).astype(int)
    return dict(acc=accuracy_score(y, pr), f1=f1_score(y, pr, zero_division=0), prec=precision_score(y, pr, zero_division=0),
                rec=recall_score(y, pr, zero_division=0), auc=roc_auc_score(y, p) if len(np.unique(y)) > 1 else float("nan"))
def pos_weight(y): return torch.tensor(float((1 - y).sum() / max(y.sum(), 1)), device=DEVICE)
def make_loader(*arrs, shuffle=True):
    ds = TensorDataset(*[torch.as_tensor(a) for a in arrs])
    return DataLoader(ds, batch_size=CFG["BATCH"], shuffle=shuffle, drop_last=shuffle and len(ds) > CFG["BATCH"])

## Stage 4: AI-based side-channel leakage detection (TCN)
Trained on Chameleon only: 5-way execution-condition classification plus a binary leakage head. Test windows come from acquisitions never seen in training.

In [ ]:
def fit_stage4(epochs):
    set_seed(); model = new_model()
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    dl = make_loader(Fc[tr_c], yc[tr_c], yl[tr_c].astype(np.float32)); pw = pos_weight(yl[tr_c])
    best, best_state, bad, hist = -1, None, 0, []
    for ep in range(epochs):
        model.train(); tot = 0; n = 0
        for xb, yb, lb in dl:
            xb, yb, lb = xb.to(DEVICE), yb.to(DEVICE), lb.to(DEVICE)
            z = model.embed(xb, "c")
            loss = F.cross_entropy(model.head_cond(z), yb) + F.binary_cross_entropy_with_logits(model.head_leak(z).squeeze(1), lb, pos_weight=pw)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
            tot += loss.item() * len(xb); n += len(xb)
        sched.step()
        pv = predict(model, Fc[va_c], "c")
        f1v = f1_score(yc[va_c], pv["cond"].argmax(1), average="macro")
        hist.append(dict(epoch=ep + 1, train_loss=tot / n, val_macro_f1=f1v))
        print(f"epoch {ep+1:02d} loss {tot/n:.4f} val macroF1 {f1v:.4f}")
        if f1v > best: best, best_state, bad = f1v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= CFG["PATIENCE"]: print("early stop"); break
    model.load_state_dict(best_state); return model, pd.DataFrame(hist)

model_s4, hist4 = fit_stage4(CFG["EPOCHS_S4"])
pt = predict(model_s4, Fc[te_c], "c")
cond_pred = pt["cond"].argmax(1); p_leak_s4 = sigmoid(pt["leak"])
print("\nStage 4 TEST (unseen acquisitions)")
print("condition accuracy:", round(accuracy_score(yc[te_c], cond_pred), 4), "| macro-F1:", round(f1_score(yc[te_c], cond_pred, average="macro"), 4))
print(classification_report(yc[te_c], cond_pred, target_names=le_cond.classes_, zero_division=0))
print("leakage head:", {k: round(v, 4) for k, v in bin_metrics(yl[te_c], p_leak_s4).items()})

# baseline: random forest on wavelet-packet statistics
rf = RandomForestClassifier(300, random_state=SEED, n_jobs=-1).fit(Sc[tr_c], yc[tr_c])
print("RF baseline (WPD stats) condition macro-F1:", round(f1_score(yc[te_c], rf.predict(Sc[te_c]), average="macro"), 4))

cm = confusion_matrix(yc[te_c], cond_pred)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(cm, cmap="Blues"); ax[0].set_xticks(range(N_COND)); ax[0].set_yticks(range(N_COND))
ax[0].set_xticklabels(le_cond.classes_); ax[0].set_yticklabels(le_cond.classes_); ax[0].set_title("Stage 4 confusion matrix")
for i in range(N_COND):
    for j in range(N_COND): ax[0].text(j, i, cm[i, j], ha="center", va="center")
ax[1].plot(hist4.epoch, hist4.train_loss, label="train loss"); ax[1].plot(hist4.epoch, hist4.val_macro_f1, label="val macro-F1"); ax[1].legend(); ax[1].set_title("Training")
plt.tight_layout(); plt.savefig(OUT / "stage4_tcn.png", dpi=130); plt.show()

## Stage 5 (Contribution 2, CDL-HMD): Domain-adversarial hardware-Trojan detection
Chameleon windows train the cryptographic side-channel encoder; UCI windows train a **separate** Trojan head with its verified labels. A gradient-reversal domain discriminator pushes the shared embedding toward measurement-source invariance.
The three task losses are combined with learned uncertainty weights (multi-task). Two variants are trained so the effect of adversarial alignment is measured, not assumed.

In [ ]:
def cyc(dl):
    while True:
        for b in dl: yield b

def fit_joint(lam_max, epochs, tag):
    set_seed(); model = new_model()
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    dl_c = make_loader(Fc[tr_c], yc[tr_c], yl[tr_c].astype(np.float32)); dl_u = make_loader(Fu[tr_u], yu[tr_u].astype(np.float32))
    ic, iu = cyc(dl_c), cyc(dl_u); steps = max(len(dl_c), len(dl_u)); total = steps * epochs
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total)
    pw_l, pw_t = pos_weight(yl[tr_c]), pos_weight(yu[tr_u]); step = 0
    best, best_state, bad, hist = -1, None, 0, []
    for ep in range(epochs):
        model.train(); agg = dict(task=0, dom=0, dacc=0)
        for _ in range(steps):
            xc, ycb, lcb = [t.to(DEVICE) for t in next(ic)]; xu, yub = [t.to(DEVICE) for t in next(iu)]
            zc, zu = model.embed(xc, "c"), model.embed(xu, "u")
            Ls = [F.cross_entropy(model.head_cond(zc), ycb),
                  F.binary_cross_entropy_with_logits(model.head_leak(zc).squeeze(1), lcb, pos_weight=pw_l),
                  F.binary_cross_entropy_with_logits(model.head_troj(zu).squeeze(1), yub, pos_weight=pw_t)]
            task = sum(torch.exp(-model.log_var[i]) * Ls[i] + model.log_var[i] for i in range(3))
            loss = task
            if lam_max > 0:
                prog = step / total; lam = lam_max * (2 / (1 + math.exp(-10 * prog)) - 1)
                dz = torch.cat([zc, zu]); dl_lab = torch.cat([torch.zeros(len(zc)), torch.ones(len(zu))]).to(DEVICE)
                dlog = model.disc(GRL.apply(dz, lam)).squeeze(1)
                ld = F.binary_cross_entropy_with_logits(dlog, dl_lab); loss = task + ld
                agg["dom"] += ld.item(); agg["dacc"] += ((dlog > 0).float() == dl_lab).float().mean().item()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step(); sched.step()
            agg["task"] += task.item(); step += 1
        pc, pu = predict(model, Fc[va_c], "c"), predict(model, Fu[va_u], "u")
        f1c = f1_score(yc[va_c], pc["cond"].argmax(1), average="macro")
        auc_l = bin_metrics(yl[va_c], sigmoid(pc["leak"]))["auc"]; auc_t = bin_metrics(yu[va_u], sigmoid(pu["troj"]))["auc"]
        score = np.nanmean([f1c, auc_l, auc_t])
        hist.append(dict(epoch=ep + 1, task_loss=agg["task"] / steps, dom_loss=agg["dom"] / steps, dom_acc=agg["dacc"] / steps,
                         val_cond_f1=f1c, val_leak_auc=auc_l, val_troj_auc=auc_t, score=score))
        print(f"[{tag}] ep {ep+1:02d} task {agg['task']/steps:.3f} dom_acc {agg['dacc']/steps:.2f} | val condF1 {f1c:.3f} leakAUC {auc_l:.3f} trojAUC {auc_t:.3f}")
        if score > best: best, best_state, bad = score, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= CFG["PATIENCE"]: print("early stop"); break
    model.load_state_dict(best_state); return model, pd.DataFrame(hist)

model_na, hist_na = fit_joint(0.0, CFG["EPOCHS_S5"], "multi-task only")
model_s5, hist_s5 = fit_joint(CFG["DANN_LAMBDA"], CFG["EPOCHS_S5"], "WACTRA (DANN)")

def mmd_rbf(A, B, n=400, seed=0):
    r = np.random.default_rng(seed); A = A[r.choice(len(A), min(n, len(A)), replace=False)]; B = B[r.choice(len(B), min(n, len(B)), replace=False)]
    Z = np.concatenate([A, B]); d2 = ((Z[:, None] - Z[None]) ** 2).sum(-1); s = np.median(d2[d2 > 0]) + 1e-8; K = np.exp(-d2 / s)
    na = len(A); return float(K[:na, :na].mean() + K[na:, na:].mean() - 2 * K[:na, na:].mean())

def domain_probe(Fa, Fb, seed=SEED):
    Xp = np.concatenate([Fa, Fb]); yp = np.r_[np.zeros(len(Fa)), np.ones(len(Fb))]
    idx = np.random.default_rng(seed).permutation(len(yp)); h = len(idx) // 2
    Xp = (Xp - Xp[idx[:h]].mean(0)) / (Xp[idx[:h]].std(0) + 1e-8)
    clf = LogisticRegression(max_iter=2000).fit(Xp[idx[:h]], yp[idx[:h]]); return clf.score(Xp[idx[h:]], yp[idx[h:]])

rows = []
for name, m in [("multi-task only (no adversarial)", model_na), ("WACTRA (domain-adversarial)", model_s5)]:
    pc, pu = predict(m, Fc[te_c], "c"), predict(m, Fu[te_u], "u")
    tm = bin_metrics(yu[te_u], sigmoid(pu["troj"])); lm = bin_metrics(yl[te_c], sigmoid(pc["leak"]))
    rows.append(dict(model=name, cond_macroF1=f1_score(yc[te_c], pc["cond"].argmax(1), average="macro"),
                     leak_AUC=lm["auc"], troj_acc=tm["acc"], troj_F1=tm["f1"], troj_AUC=tm["auc"],
                     embedding_domain_probe_acc=domain_probe(pc["emb"], pu["emb"]), MMD2=mmd_rbf(pc["emb"], pu["emb"])))
rf_u = RandomForestClassifier(300, random_state=SEED, n_jobs=-1).fit(Su[tr_u], yu[tr_u])
rows.append(dict(model="RF baseline (WPD stats, UCI only)", troj_acc=accuracy_score(yu[te_u], rf_u.predict(Su[te_u])),
                 troj_F1=f1_score(yu[te_u], rf_u.predict(Su[te_u])), troj_AUC=roc_auc_score(yu[te_u], rf_u.predict_proba(Su[te_u])[:, 1])))
raw_probe = domain_probe(Sc[:, :NB], Su[:, :NB])
tab5 = pd.DataFrame(rows).round(4); display(tab5)
print(f"Raw WPD-energy domain probe accuracy (before encoder): {raw_probe:.3f}  (1.0 = datasets trivially separable; 0.5 = indistinguishable)")
tab5.to_csv(OUT / "stage5_comparison.csv", index=False)

## Stage 6: Joint security decision and threat categorization
Heads stay separate. Logits are temperature-calibrated on validation data, per-head decision thresholds and an abstention band are also fitted on validation data only.
A joint decision is `normal`, `leakage`, `hardware_modification`, `both`, or `uncertain` (when a head is missing or inside its abstention band).
Because the two datasets come from different devices, evaluation pairs are **independent evidence pairs** (a Chameleon window + a UCI window); this is a protocol for exercising the decision logic, not a claim that the two measurements come from the same chip.

In [ ]:
def fit_temperature(logits, y):
    lg = torch.tensor(logits, dtype=torch.float32); yy = torch.tensor(y, dtype=torch.float32)
    T = torch.nn.Parameter(torch.ones(1)); opt = torch.optim.LBFGS([T], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad(); loss = F.binary_cross_entropy_with_logits(lg / T.clamp(min=0.05), yy); loss.backward(); return loss
    opt.step(closure); return float(T.clamp(min=0.05).item())

def ece_binary(p, y, bins=10):
    conf = np.maximum(p, 1 - p); acc = ((p >= 0.5).astype(int) == y); edges = np.linspace(0.5, 1, bins + 1); e = 0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

def best_f1_threshold(p, y):
    grid = np.linspace(0.05, 0.95, 91); f = [f1_score(y, (p >= t).astype(int), zero_division=0) for t in grid]
    return float(grid[int(np.argmax(f))])

def fit_abstain_band(p, y, tau, target=0.95):
    best = (0.0, -1.0)
    for d in np.linspace(0, 0.45, 46):
        keep = np.abs(p - tau) >= d
        if keep.sum() < 0.5 * len(p): break
        acc = ((p[keep] >= tau).astype(int) == y[keep]).mean()
        if acc >= target: return float(d)
        if acc > best[1]: best = (float(d), acc)
    return best[0]

pcv, puv = predict(model_s5, Fc[va_c], "c"), predict(model_s5, Fu[va_u], "u")
pct, put = predict(model_s5, Fc[te_c], "c"), predict(model_s5, Fu[te_u], "u")
T_leak = fit_temperature(pcv["leak"], yl[va_c]); T_troj = fit_temperature(puv["troj"], yu[va_u])
pl_val, pt_val = sigmoid(pcv["leak"] / T_leak), sigmoid(puv["troj"] / T_troj)
pl_te, pt_te = sigmoid(pct["leak"] / T_leak), sigmoid(put["troj"] / T_troj)
tau_l, tau_t = best_f1_threshold(pl_val, yl[va_c]), best_f1_threshold(pt_val, yu[va_u])
d_l, d_t = fit_abstain_band(pl_val, yl[va_c], tau_l), fit_abstain_band(pt_val, yu[va_u], tau_t)
print(f"Temperature  leak {T_leak:.3f} | troj {T_troj:.3f}")
print(f"ECE test     leak {ece_binary(sigmoid(pct['leak']), yl[te_c]):.4f} -> {ece_binary(pl_te, yl[te_c]):.4f} | troj {ece_binary(sigmoid(put['troj']), yu[te_u]):.4f} -> {ece_binary(pt_te, yu[te_u]):.4f}")
print(f"Thresholds   leak tau={tau_l:.2f} band={d_l:.2f} | troj tau={tau_t:.2f} band={d_t:.2f}")

def head_state(p, tau, d):
    if p is None or abs(p - tau) < d: return None
    return int(p >= tau)

def decide(pl, pt_):
    sl, st = head_state(pl, tau_l, d_l), head_state(pt_, tau_t, d_t)
    if sl == 1 and st == 1: return "both", "full"
    if sl == 0 and st == 0: return "normal", "full"
    if sl == 1 and st == 0: return "leakage", "full"
    if sl == 0 and st == 1: return "hardware_modification", "full"
    if sl == 1 and st is None: return "leakage", "partial"
    if st == 1 and sl is None: return "hardware_modification", "partial"
    return "uncertain", "none"

CATS = ["normal", "leakage", "hardware_modification", "both"]
truth_cat = lambda l, t: "both" if l and t else "leakage" if l else "hardware_modification" if t else "normal"
rng = np.random.default_rng(SEED); n_pairs = 2000
ci, uj = rng.integers(0, len(te_c), n_pairs), rng.integers(0, len(te_u), n_pairs)
pred = [decide(pl_te[a], pt_te[b])[0] for a, b in zip(ci, uj)]
true = [truth_cat(yl[te_c][a], yu[te_u][b]) for a, b in zip(ci, uj)]
pred_a, true_a = np.array(pred), np.array(true); cov = pred_a != "uncertain"
print(f"\nJoint decision on {n_pairs} evidence pairs: coverage {cov.mean():.3f} | accuracy (all, uncertain=wrong) {np.mean(pred_a == true_a):.3f} | accuracy (covered) {np.mean(pred_a[cov] == true_a[cov]):.3f}")
labels6 = CATS + ["uncertain"]
cm6 = pd.DataFrame(confusion_matrix(true_a, pred_a, labels=labels6), index=[f"true:{c}" for c in labels6], columns=[f"pred:{c}" for c in labels6]); display(cm6)

ps = [decide(pl_te[a], None)[0:2] for a in range(len(te_c))]; pu_ = [decide(None, pt_te[b])[0:2] for b in range(len(te_u))]
print("Chameleon-only evidence (Trojan head missing):", pd.Series([f"{a}/{b}" for a, b in ps]).value_counts().to_dict())
print("UCI-only evidence (leakage head missing)     :", pd.Series([f"{a}/{b}" for a, b in pu_]).value_counts().to_dict())

## Stage 7 (Contribution 3, XH-TRA): Explainable threat attribution and risk assessment
SHAP is applied to **wavelet-packet subband groups** (each subband channel is one SHAP feature; "absent" subbands are replaced by the training mean), so attributions are directly interpretable in frequency terms.
Then: cross-dataset importance comparison, deletion-fidelity tests (does removing top-ranked subbands hurt more than removing random/low-ranked ones?), and calibrated, threat-specific risk scores.

In [ ]:
model_s5.eval()
bg_c = Fc[tr_c].mean(0).astype(np.float32); bg_u = Fu[tr_u].mean(0).astype(np.float32)
names_c = [f"Power/{BAND_NAMES[b]}" for b in range(NB)]
names_u = [f"Power/{BAND_NAMES[b]}" for b in range(NB)] + [f"EM/{BAND_NAMES[b]}" for b in range(NB)]

def group_predictor(x, bg, domain, head, T):
    xt = torch.as_tensor(x[None]).to(DEVICE); bt = torch.as_tensor(bg[None]).to(DEVICE); H = getattr(model_s5, "head_" + head)
    def f(Z):
        Z = np.asarray(Z, dtype=np.float32); out = []
        for i in range(0, len(Z), 256):
            zb = torch.as_tensor(Z[i:i + 256]).to(DEVICE)[:, :, None]
            with torch.no_grad(): lg = H(model_s5.embed(zb * xt + (1 - zb) * bt, domain)).squeeze(1)
            out.append(torch.sigmoid(lg / T).cpu().numpy())
        return np.concatenate(out)
    return f

def group_importance(x, bg, domain, head, T, nsamples):
    C = x.shape[0]; f = group_predictor(x, bg, domain, head, T)
    if HAS_SHAP:
        ex = shap.KernelExplainer(f, np.zeros((1, C), dtype=np.float32))
        sv = ex.shap_values(np.ones((1, C), dtype=np.float32), nsamples=nsamples, silent=True)
        return np.asarray(sv).reshape(-1)[:C]
    base = f(np.ones((1, C), np.float32))[0]; imp = np.zeros(C)             # occlusion fallback
    for c in range(C):
        z = np.ones((1, C), np.float32); z[0, c] = 0; imp[c] = base - f(z)[0]
    return imp

r = np.random.default_rng(SEED)
sel_c = r.choice(len(te_c), min(40, len(te_c)), replace=False); sel_u = r.choice(len(te_u), min(30, len(te_u)), replace=False)
t0 = time.time()
S_leak = np.array([group_importance(Fc[te_c][i], bg_c, "c", "leak", T_leak, 2 ** 8) for i in sel_c])
S_troj = np.array([group_importance(Fu[te_u][i], bg_u, "u", "troj", T_troj, 400) for i in sel_u])
print(f"attribution done in {time.time()-t0:.0f}s | method: {'SHAP KernelExplainer (subband groups)' if HAS_SHAP else 'occlusion fallback'}")
imp_leak, imp_troj = np.abs(S_leak).mean(0), np.abs(S_troj).mean(0)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].bar(range(len(imp_leak)), imp_leak); ax[0].set_xticks(range(len(imp_leak))); ax[0].set_xticklabels([f"B{b}" for b in range(NB)])
ax[0].set_title("Leakage head (Chameleon): mean |SHAP| per power subband")
ax[1].bar(range(len(imp_troj)), imp_troj, color=["tab:blue"] * NB + ["tab:orange"] * NB); ax[1].set_xticks(range(len(imp_troj)))
ax[1].set_xticklabels([n.split(" ")[0] for n in names_u], rotation=90); ax[1].set_title("Trojan head (UCI): mean |SHAP| per power(blue)/EM(orange) subband")
plt.tight_layout(); plt.savefig(OUT / "c3_shap_importance.png", dpi=130); plt.show()

# --- cross-dataset comparison of importance patterns ---
rho_pow = spearmanr(imp_leak, imp_troj[:NB]).correlation
print(f"Spearman(leak importance on Chameleon power, Trojan importance on UCI power) = {rho_pow:.3f}")
print("Top leakage subbands :", [names_c[i].split(' ')[0] + ' ' + names_c[i].split('(')[1][:-1] for i in np.argsort(-imp_leak)[:3]])
print("Top Trojan subbands  :", [names_u[i].split(' ')[0] + ' ' + names_u[i].split('(')[1][:-1] for i in np.argsort(-imp_troj)[:3]])
print("Power vs EM share of Trojan importance:", round(imp_troj[:NB].sum() / imp_troj.sum(), 3), "/", round(imp_troj[NB:].sum() / imp_troj.sum(), 3))

In [ ]:
# --- deletion fidelity: meaningful signal use vs dataset artifacts ---
def masked_auc(X, y, domain, head, T, mask_idx, bg):
    Xm = X.copy()
    if len(mask_idx): Xm[:, list(mask_idx), :] = bg[list(mask_idx)][None]
    p = sigmoid(predict(model_s5, Xm, domain)[head] / T); return roc_auc_score(y, p)

def fidelity(X, y, domain, head, T, imp, bg, k=3, reps=10):
    order = np.argsort(-imp); rr = np.random.default_rng(SEED)
    return dict(full=masked_auc(X, y, domain, head, T, [], bg), remove_top_k=masked_auc(X, y, domain, head, T, order[:k], bg),
                remove_bottom_k=masked_auc(X, y, domain, head, T, order[-k:], bg),
                remove_random_k=float(np.mean([masked_auc(X, y, domain, head, T, rr.choice(len(imp), k, replace=False), bg) for _ in range(reps)])))
fid = pd.DataFrame({"leakage head (Chameleon)": fidelity(Fc[te_c], yl[te_c], "c", "leak", T_leak, imp_leak, bg_c),
                    "Trojan head (UCI)": fidelity(Fu[te_u], yu[te_u], "u", "troj", T_troj, imp_troj, bg_u)}).T.round(4)
display(fid)
print("Reading: a faithful, signal-driven model loses much more AUC when top-ranked subbands are removed than when bottom-ranked or random ones are.")

# --- artifact check: can a linear probe tell the datasets apart? ---
print(f"Dataset-identity probe  raw WPD energies: {raw_probe:.3f} | encoder embeddings (no adversarial): {tab5.iloc[0].embedding_domain_probe_acc:.3f} | (adversarial): {tab5.iloc[1].embedding_domain_probe_acc:.3f}")

# --- threat-specific risk scores and per-sample explanations ---
def top_features(S, names, i, k=3):
    o = np.argsort(-np.abs(S[i]))[:k]; return "; ".join(f"{names[j].split(' ')[0]}({S[i][j]:+.3f})" for j in o)

def risk_scores(pl, pt_):
    rl = pl * CFG["SEVERITY"]["leakage"]; rt = pt_ * CFG["SEVERITY"]["hardware_modification"]
    return rl, rt, 1 - (1 - rl) * (1 - rt)

n_show = min(12, len(sel_c), len(sel_u)); rows = []
for k in range(n_show):
    a, b = sel_c[k], sel_u[k]; pl, pt_ = float(pl_te[a]), float(pt_te[b]); cat, ev = decide(pl, pt_); rl, rt, rc = risk_scores(pl, pt_)
    rows.append(dict(case=k, predicted=cat, evidence=ev, confidence=round(float(np.mean([max(pl, 1 - pl), max(pt_, 1 - pt_)])), 3),
                     p_leakage=round(pl, 3), p_hw_mod=round(pt_, 3), risk_leakage=round(rl, 3), risk_hw_mod=round(rt, 3), risk_combined=round(rc, 3),
                     truth=truth_cat(yl[te_c][a], yu[te_u][b]), top_leakage_features=top_features(S_leak, names_c, k), top_hw_mod_features=top_features(S_troj, names_u, k)))
assessment = pd.DataFrame(rows); pd.set_option("display.max_colwidth", 80); display(assessment)
assessment.to_csv(OUT / "xh_tra_security_assessment.csv", index=False)

## Save artifacts

In [ ]:
torch.save(dict(stage4=model_s4.state_dict(), joint_no_adv=model_na.state_dict(), wactra=model_s5.state_dict()), OUT / "wactra_models.pt")
summary = dict(
    data_sources=dict(chameleon=raw_cham["source"], uci_ht=raw_uci["source"]),
    stage4=dict(cond_acc=float(accuracy_score(yc[te_c], cond_pred)), cond_macro_f1=float(f1_score(yc[te_c], cond_pred, average="macro")), leak=bin_metrics(yl[te_c], p_leak_s4)),
    stage5=tab5.to_dict(orient="records"),
    stage6=dict(T_leak=T_leak, T_troj=T_troj, tau_leak=tau_l, tau_troj=tau_t, band_leak=d_l, band_troj=d_t,
                coverage=float(cov.mean()), acc_covered=float(np.mean(pred_a[cov] == true_a[cov]))),
    stage7=dict(spearman_power=float(rho_pow), top_leak_bands=[int(i) for i in np.argsort(-imp_leak)[:3]], top_troj=[names_u[i] for i in np.argsort(-imp_troj)[:3]]))
json.dump(summary, open(OUT / "wactra_summary.json", "w"), indent=2, default=float)
print(json.dumps(summary, indent=2, default=float)[:1500]); print("\nSaved to", OUT.resolve())
if "SYNTHETIC" in (raw_cham["source"], raw_uci["source"]):
    print("\nNOTE: at least one dataset was SYNTHETIC. Place the real Chameleon / UCI files in the configured folders and re-run for publishable results.")